# Data Wrangling — Sesi 5: Transformasi Data
Notebook praktikum. Dikerjakan berpasangan: satu orang menarasikan, satu mengetik; bertukar peran di menit 71.

Dataset: `ecommerce_transactions.csv` (Materi Sesi 2 di RISE). Unggah ke Colab (ikon folder di kiri, lalu *Upload*) sebelum menjalankan sel pertama.

Kumpulkan notebook ini (File → Download → .ipynb) di RISE Tugas Sesi 5 **sebelum menit 80 (19:50 WIB)**.

---

**CONTOH OUTPUT PARSIAL** — notebook ini contoh pengumpulan yang berhenti di tengah lab (bukan kunci jawaban). Sel yang sudah dijalankan memakai dataset yang sama dengan kelasmu, jadi angkanya harus cocok. Di luar itu: tabel keputusan baru terisi 2 dari 4 baris, dan pertanyaan penutup masih kosong — bagian itulah tugasmu.

*Dicetak dengan pandas 3.0.6 — di Colab (pandas 2.x), dtype kolom teks tampil* `object`*, bukan* `str`*.*

## 0. Siapkan data

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("ecommerce_transactions.csv")
print(df.shape)
df.dtypes

(500, 10)


order_id              str
date                  str
product               str
category              str
price               int64
qty               float64
customer_id           str
city                  str
payment_method        str
discount            int64

Langkah-langkah persiapan (hasil Sesi 3 dan slide 6):
1. hapus duplikat,
2. sisihkan harga ≤ 0 (harga negatif tidak bermakna sebagai skala),
3. rapikan huruf dan spasi pada `city` dan `product`.

In [2]:
df = df.drop_duplicates()
df = df[df["price"] > 0].copy()
df["city"] = df["city"].str.strip().str.title()
df["product"] = df["product"].str.strip().str.title()

# Periksa: angka ini harus cocok dengan slide
print(len(df), df["city"].nunique(), df["product"].nunique())
assert (len(df), df["city"].nunique(), df["product"].nunique()) == (487, 8, 24)

487 8 24


## 1. Scaling `price`
Min-max: x' = (x − min) / (maks − min). Z-score: z = (x − rata-rata) / simpangan baku.

In [3]:
p = df["price"]
df["price_minmax"] = (p - p.min()) / (p.max() - p.min())
df["price_z"] = (p - p.mean()) / p.std()

print(df["price_minmax"].min(), df["price_minmax"].max())
print(round(df["price_z"].mean(), 6), round(df["price_z"].std(), 6))
print("baris di bawah 0,1:", (df["price_minmax"] < 0.1).sum(), "dari", len(df))

0.0 1.0
0.0 1.0
baris di bawah 0,1: 474 dari 487


Versi scikit-learn. Perhatikan: `StandardScaler` memakai pembagi n, pandas `std()` memakai n−1, jadi angkanya sedikit berbeda.

*_(contoh ini berhenti di sini — sel di bawah belum dijalankan)_*

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

mm = MinMaxScaler().fit_transform(df[["price"]]).ravel()
zz = StandardScaler().fit_transform(df[["price"]]).ravel()
print(np.allclose(mm, df["price_minmax"]))
print(np.abs(zz - df["price_z"]).max())

**Pertanyaan 1.** Setelah min-max, di mana sebagian besar harga berada? Tulis satu kalimat.

_jawaban Anda:_

## 2. One-hot `category` dan `payment_method`

In [ ]:
enc = pd.get_dummies(df, columns=["category", "payment_method"])
baru = [c for c in enc.columns if c.startswith(("category_", "payment_method_"))]
print(len(baru), baru)
assert len(baru) == 9

Nilai kosong pada `payment_method` (Sesi 4) menghasilkan baris dengan semua kolom 0. Itu wajar bila belum diisi; bila sudah diimputasi, tidak ada baris seperti itu.

## 3. Tabel keputusan
Isi empat baris. Untuk tiap kolom: transformasi apa (atau **tidak** ada), dan satu kalimat alasan. Minimal dua baris menjelaskan *mengapa tidak* melakukan sesuatu.

_(contoh ini baru mengisi dua baris — lengkapi sisanya)_

| Kolom | Jenis | Transformasi | Alasan |
| :---- | :---- | :---- | :---- |
| city | kategorik | one-hot (8 kolom) | tidak berurutan; sudah dirapikan di atas |
| product | kategorik | tidak di-one-hot | 24 nilai terlalu banyak; kelompokkan atau pakai frekuensi |
| discount |  |  |  |
| customer_id |  |  |  |

**Penutup:** satu kalimat, bagaimana Anda memakai ini pada data kerja Anda hari Senin.

_jawaban Anda:_